In [1]:
import pandas as pd

# Load the RAW Kaggle file (we never modify the original - golden rule!)
df = pd.read_csv("../data/raw/train.csv")

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [3]:
print("COLUMNS:", df.columns.tolist())
print("SHAPE (rows, cols):", df.shape)
print("\nDATA TYPES:\n", df.dtypes)
print("\nMISSING VALUES:\n", df.isnull().sum())
print("\nDUPLICATE ROWS:", df.duplicated().sum())

COLUMNS: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales']
SHAPE (rows, cols): (9800, 18)

DATA TYPES:
 Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code      float64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
dtype: object

MISSING VALUES:
 Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      1

In [12]:
df = pd.read_csv("../data/raw/train.csv")
# Fix the date parsing (format='mixed' and dayfirst=True handles the Kaggle mess)
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed', dayfirst=True, errors='coerce')

print("Total Rows Recovered:", len(df))
print("Bad Dates Remaining:", df['Order Date'].isna().sum())

Total Rows Recovered: 9800
Bad Dates Remaining: 0


In [13]:
# Rename Kaggle columns -> workbook column names (only if they exist)
rename_map = {
    'Order ID': 'Order_ID',
    'Order Date': 'Order_Date',
    'Customer ID': 'Customer_ID',
    'Product Name': 'Product',
    'Sales': 'Revenue',
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
print(df.columns.tolist())

['Row ID', 'Order_ID', 'Order_Date', 'Ship Date', 'Ship Mode', 'Customer_ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product', 'Revenue']


In [14]:
print("ROWS BEFORE CLEANING:", len(df))
print("\nMISSING VALUES:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nFULL DUPLICATE ROWS:", df.duplicated().sum())
if 'Order_ID' in df.columns:
    print("DUPLICATE ORDER IDs:", df['Order_ID'].duplicated().sum())
if 'Order_Date' in df.columns:
    dates = pd.to_datetime(df['Order_Date'], errors='coerce')
    print("\nIMPOSSIBLE/UNPARSEABLE DATES:", dates.isna().sum())
    print("DATE RANGE:", dates.min(), "to", dates.max())
for col in ['Quantity', 'Revenue']:
    if col in df.columns:
        num = pd.to_numeric(df[col], errors='coerce')
        print(f"\n{col} <= 0 (data-entry errors):", (num <= 0).sum())
print("\nTEXT CONSISTENCY CHECK:")
for col in ['Category', 'Region']:
    if col in df.columns:
        print(col, "->", sorted(df[col].dropna().astype(str).unique()))

ROWS BEFORE CLEANING: 9800

MISSING VALUES:
 Postal Code    11
dtype: int64

FULL DUPLICATE ROWS: 0
DUPLICATE ORDER IDs: 4878

IMPOSSIBLE/UNPARSEABLE DATES: 0
DATE RANGE: 2015-01-03 00:00:00 to 2018-12-30 00:00:00

Revenue <= 0 (data-entry errors): 0

TEXT CONSISTENCY CHECK:
Category -> ['Furniture', 'Office Supplies', 'Technology']
Region -> ['Central', 'East', 'South', 'West']


In [15]:
df_clean = df.copy()

# 1. Dates: convert properly, drop impossible dates
df_clean['Order_Date'] = pd.to_datetime(df_clean['Order_Date'], errors='coerce')
n_bad_dates = df_clean['Order_Date'].isna().sum()
df_clean = df_clean.dropna(subset=['Order_Date'])

# 2. Numbers stored as text -> force numeric, drop invalids
for col in ['Quantity', 'Revenue', 'Profit']:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
num_cols = [c for c in ['Quantity', 'Revenue', 'Profit'] if c in df_clean.columns]
df_clean = df_clean.dropna(subset=num_cols)

# 3. Negative/zero quantity or revenue = entry errors -> drop
mask_bad = False
if 'Quantity' in df_clean.columns: mask_bad = mask_bad | (df_clean['Quantity'] <= 0)
if 'Revenue' in df_clean.columns:  mask_bad = mask_bad | (df_clean['Revenue'] <= 0)
n_bad_rows = mask_bad.sum()
df_clean = df_clean[~mask_bad]

# 4. Duplicates
n_dup = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()

# 5. Text hygiene: trim spaces, consistent capitalisation (safe columns only)
for col in ['Category', 'Region']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
for col in ['Product', 'Customer_ID']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip()

print("Dropped bad dates:", n_bad_dates)
print("Dropped bad qty/revenue:", n_bad_rows)
print("Removed duplicates:", n_dup)
print("ROWS AFTER CLEANING:", len(df_clean))

Dropped bad dates: 0
Dropped bad qty/revenue: 0
Removed duplicates: 0
ROWS AFTER CLEANING: 9800


In [16]:
df_clean = df.copy()

# 1. Dates: convert properly, drop impossible dates
df_clean['Order_Date'] = pd.to_datetime(df_clean['Order_Date'], errors='coerce')
n_bad_dates = df_clean['Order_Date'].isna().sum()
df_clean = df_clean.dropna(subset=['Order_Date'])

# 2. Numbers stored as text -> force numeric, drop invalids
for col in ['Quantity', 'Revenue', 'Profit']:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
num_cols = [c for c in ['Quantity', 'Revenue', 'Profit'] if c in df_clean.columns]
df_clean = df_clean.dropna(subset=num_cols)

# 3. Negative/zero quantity or revenue = entry errors -> drop
mask_bad = False
if 'Quantity' in df_clean.columns: mask_bad = mask_bad | (df_clean['Quantity'] <= 0)
if 'Revenue' in df_clean.columns:  mask_bad = mask_bad | (df_clean['Revenue'] <= 0)
n_bad_rows = mask_bad.sum()
df_clean = df_clean[~mask_bad]

# 4. Duplicates
n_dup = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()

# 5. Text hygiene: trim spaces, consistent capitalisation (safe columns only)
for col in ['Category', 'Region']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
for col in ['Product', 'Customer_ID']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip()

print("Dropped bad dates:", n_bad_dates)
print("Dropped bad qty/revenue:", n_bad_rows)
print("Removed duplicates:", n_dup)
print("ROWS AFTER CLEANING:", len(df_clean))

Dropped bad dates: 0
Dropped bad qty/revenue: 0
Removed duplicates: 0
ROWS AFTER CLEANING: 9800


In [17]:
# Time columns for trend analysis
df_clean['Year'] = df_clean['Order_Date'].dt.year
df_clean['Month_Name'] = df_clean['Order_Date'].dt.month_name()
df_clean['Month_Year'] = df_clean['Order_Date'].dt.to_period('M').astype(str)

# Customer_Segment: New vs Repeat (based on unique orders per customer)
order_counts = df_clean.groupby('Customer_ID')['Order_ID'].transform('nunique')
df_clean['Customer_Segment'] = 'New'
df_clean.loc[order_counts > 1, 'Customer_Segment'] = 'Repeat'

# Order value buckets: Low / Medium / High
df_clean['Order_Value_Category'] = pd.qcut(
    df_clean['Revenue'], q=3, labels=['Low', 'Medium', 'High'], duplicates='drop'
).astype(str)

print(df_clean['Customer_Segment'].value_counts())
print(df_clean['Order_Value_Category'].value_counts())

Customer_Segment
Repeat    9772
New         28
Name: count, dtype: int64
Order_Value_Category
Low       3267
Medium    3267
High      3266
Name: count, dtype: int64


In [18]:
df_clean.to_csv('../data/processed/superstore_clean.csv', index=False)
print("Saved. Final shape:", df_clean.shape)

Saved. Final shape: (9800, 23)
